In [14]:
# Script: Extração de voltas (laps) FastF1 2022–2024      # Descrição breve
# -*- coding: utf-8 -*-                                   # Codificação do arquivo (acentos PT-BR)
# Objetivo: Usar cache SOMENTE em notebooks/fastf1_cache  # Padronização do cache

from pathlib import Path                                   # Manipulação de caminhos de forma portátil
import pandas as pd                                        # DataFrames e IO
import numpy as np                                         # Utilidades numéricas
import fastf1                                             # Biblioteca FastF1 (dados de F1)

# =========================
# Parâmetros gerais
# =========================

YEARS = [2022, 2023, 2024]                                 # Anos-alvo para extração
SKIP_IF_SAVED = True                                       # Se True, não reextrai quando já existe arquivo salvo

# =========================
# Estrutura de pastas (sempre usar notebooks/fastf1_cache)
# =========================
CWD = Path.cwd().resolve()                                 # Diretório corrente absoluto
if CWD.name.lower() == "notebooks":                        # Caso o notebook esteja dentro de /notebooks
    NB_DIR = CWD                                           # NB_DIR = .../notebooks
else:                                                      # Caso o notebook rode a partir da raiz do repositório
    NB_DIR = CWD / "notebooks"                             # NB_DIR = .../notebooks (força o uso dessa pasta)

DATA_DIR = NB_DIR / "data"                                 # Saídas e dados em notebooks/data
DATA_DIR.mkdir(parents=True, exist_ok=True)                # Garante que a pasta exista

INTERIM = DATA_DIR / "interim"                             # Subpasta para arquivos intermediários
INTERIM.mkdir(parents=True, exist_ok=True)                 # Garante que a pasta exista

CACHE_DIR = NB_DIR / "fastf1_cache"                        # **ÚNICO** cache suportado: notebooks/fastf1_cache
CACHE_DIR.mkdir(parents=True, exist_ok=True)               # Garante que a pasta exista
fastf1.Cache.enable_cache(str(CACHE_DIR))                  # Ativa cache do FastF1 nesse caminho

print(f"Cache ativo em: {CACHE_DIR}")                      # Loga caminho do cache efetivo
print(f"Saídas em:      {INTERIM}")                        # Loga caminho das saídas

# =========================
# Circuitos-alvo (mapeamento por Location e EventName)
# =========================
TARGETS = {
    # Já existentes
    "Bahrein":     {"location": ["sakhir", "bahrain"],            "event_sub": ["bahrain"]},
    "Jeddah":      {"location": ["jeddah"],                       "event_sub": ["saudi"]},
    "Australia":   {"location": ["melbourne", "australia"],       "event_sub": ["australian"]},
    "Baku":        {"location": ["baku"],                         "event_sub": ["azerbaijan"]},
    "Miami":       {"location": ["miami"],                        "event_sub": ["miami"]},
    "Monza":       {"location": ["monza"],                        "event_sub": ["italian"]},
    "Singapura":   {"location": ["singapore"],                    "event_sub": ["singapore"]},
    "Suzuka":      {"location": ["suzuka"],                       "event_sub": ["japan"]},
    "COTA":        {"location": ["austin"],                       "event_sub": ["united states"]},   # COTA/Austin (EUA)
    "Mexico":      {"location": ["mexico city"],                  "event_sub": ["mexico"]},
    "Brasil":      {"location": ["são paulo", "sao paulo"],       "event_sub": ["sao paulo", "brazil"]},
    "Abu Dhabi":   {"location": ["abu dhabi", "yas marina"],      "event_sub": ["abu dhabi"]},
    "Silverstone": {"location": ["silverstone"],                  "event_sub": ["british", "great britain"]},
    "Bélgica":     {"location": ["spa-francorchamps", "spa"],     "event_sub": ["belgian"]},
    "Hungria":     {"location": ["hungaroring", "budapest"],      "event_sub": ["hungarian"]},
    "Mônaco":      {"location": ["monaco"],                       "event_sub": ["monaco"]},
}

# =========================
# Carrega calendários (sem testes)
# =========================
all_events = []                                             # Buffer para todos os anos

for y in YEARS:                                             # Itera 2022, 2023, 2024
    cal = fastf1.get_event_schedule(y, include_testing=False).copy()  # Busca calendário e copia
    keep_cols = [c for c in ["RoundNumber","EventName","OfficialEventName","EventDate","Location","EventFormat"] if c in cal.columns]
                                                              # Mantém apenas colunas relevantes (as que existirem)
    cal = cal[keep_cols]                                    # Aplica o filtro de colunas
    cal["Year"] = y                                         # Marca o ano
    cal["loc_lc"] = cal["Location"].fillna("").str.lower()  # Normaliza Location (minúsculas) p/ matching
    cal["ename_lc"] = cal["EventName"].fillna("").str.lower()  # Normaliza EventName (minúsculas) p/ matching
    all_events.append(cal)                                  # Acumula esse calendário

events_df = pd.concat(all_events, ignore_index=True)        # Concatena calendários dos 3 anos em um DF

# =========================
# Função: regra de matching (linha do calendário ↔ circuito)
# =========================
def row_matches_target(row, target) -> bool:                # Retorna True se a linha pertence ao circuito-alvo
    loc = row["loc_lc"]                                     # Texto de Location em minúsculas
    enm = row["ename_lc"]                                   # Texto de EventName em minúsculas
    if any(sub in loc for sub in target["location"]):       # Bate por fragmentos na Location
        return True                                         # Se achou, casa
    if any(sub in enm for sub in target["event_sub"]):      # Ou bate por fragmentos no EventName
        return True                                         # Se achou, casa
    return False                                            # Caso contrário, não casa

# =========================
# Função: extrai voltas da corrida (session = 'R')
# =========================
def extract_event_race_laps(year: int, round_number: int | None, event_name: str | None) -> pd.DataFrame:
    """Carrega a sessão 'R' e retorna laps enriquecidas com clima (TrackTemp etc.)."""
    # 1) Buscar a sessão da corrida
    if pd.notna(round_number):
        ses = fastf1.get_session(int(year), int(round_number), 'R')
    else:
        ses = fastf1.get_session(int(year), str(event_name), 'R')

    # 2) Carregar com dados de clima
    ses.load(telemetry=False, weather=True)  # garante weather_data disponível

    # 3) Laps base
    laps = ses.laps.copy()

    # 4) Se não houver weather_data, retorna apenas laps com metadados mínimos
    if getattr(ses, "weather_data", None) is None or ses.weather_data.empty:
        laps["Year"] = year
        laps["RoundNumber"] = int(round_number) if pd.notna(round_number) else np.nan
        laps["EventName"] = ses.event.EventName if hasattr(ses, "event") else event_name
        laps["SessionName"] = "Race"
        return laps

    # 5) Preparar chave de junção temporal: meio da volta
    #    (LapStartTime e LapTime são timedeltas relativos ao início da sessão)
    if {"LapStartTime", "LapTime"}.issubset(laps.columns):
        laps["LapMidTime"] = laps["LapStartTime"] + (laps["LapTime"] / 2)
    else:
        # fallback: se faltar algo, usa LapStartTime
        laps["LapMidTime"] = laps.get("LapStartTime", pd.Series(pd.Timedelta(0), index=laps.index))

    # 6) Selecionar colunas de clima úteis
    wd = ses.weather_data.copy()
    keep_weather = [c for c in ["Time","AirTemp","TrackTemp","Humidity","Pressure","WindSpeed","WindDirection","Rainfall"] if c in wd.columns]
    wd = wd[keep_weather].sort_values("Time")

    # 7) merge_asof: associa a medição de clima mais recente até o meio da volta
    #    (usa 'backward'; se preferir a mais próxima, pode usar 'nearest')
    merged = pd.merge_asof(
        left=laps.sort_values("LapMidTime"),
        right=wd,
        left_on="LapMidTime",
        right_on="Time",
        direction="backward"
    )

    # 8) Limpeza/renomes
    merged = merged.rename(columns={"Time": "WeatherTime"})  # timestamp da amostra climática usada

    # 9) Metadados do evento
    merged["Year"] = year
    merged["RoundNumber"] = int(round_number) if pd.notna(round_number) else np.nan
    merged["EventName"] = ses.event.EventName if hasattr(ses, "event") else event_name
    merged["SessionName"] = "Race"

    return merged
                                         # Retorna DF

# =========================
# Loop principal: para cada circuito, extrair/ler e salvar
# =========================
all_circuits_laps = []                                      # Para montar o dataset mestre (todos circuitos)

for circuito, matchers in TARGETS.items():                  # Itera pelos 16 circuitos
    safe_circuit = circuito.lower().replace(" ", "_")       # Nome “seguro” para arquivo
    out_csv  = INTERIM / f"{safe_circuit}_2022-2024_all_laps.csv"      # Caminho CSV de saída
    out_parq = INTERIM / f"{safe_circuit}_2022-2024_all_laps.parquet"  # Caminho Parquet de saída

    # --- ETAPA 0: pular se já existe (não chama API) ---
    if SKIP_IF_SAVED and (out_parq.exists() or out_csv.exists()):      # Se já existe saída salva
        try:                                                           # Tenta ler
            if out_parq.exists():                                      # Prefere Parquet (rápido/leve)
                laps_circuito = pd.read_parquet(out_parq)              # Lê Parquet
                print(f"[PULADO] {circuito}: carregado de Parquet existente.")  # Log
            else:                                                      # Senão, lê CSV
                laps_circuito = pd.read_csv(out_csv)                   # Lê CSV
                print(f"[PULADO] {circuito}: carregado de CSV existente.")      # Log
            all_circuits_laps.append(laps_circuito)                    # Acumula no mestre
            continue                                                   # Vai ao próximo circuito
        except Exception as e:                                         # Se falhar leitura
            print(f"[AVISO] Falha ao ler saída existente de '{circuito}' ({e}). Reextraindo...")  # Avisa e segue

    # --- ETAPA 1: localizar eventos do circuito no calendário ---
    mask = events_df.apply(lambda r: row_matches_target(r, matchers), axis=1)  # Aplica matching linha a linha
    cal_hits = events_df[mask].copy().sort_values(["Year","RoundNumber","EventDate"])  # Eventos encontrados ordenados

    if cal_hits.empty:                                                # Se não achou nada
        print(f"[AVISO] Nenhum evento encontrado para '{circuito}' nos anos {YEARS}.")  # Loga aviso
        continue                                                      # Próximo circuito

    print(f"\n=== {circuito}: {len(cal_hits)} evento(s) 2022–2024 ===")  # Log informativo
    per_circuit_laps = []                                             # Buffer de voltas desse circuito
    errors = []                                                       # Buffer de erros (se houver)

    # --- ETAPA 2: para cada evento (ano), extrair voltas da corrida ---
    for _, ev in cal_hits.iterrows():                                 # Itera pelos eventos localizados
        y = int(ev["Year"])                                           # Ano do evento
        rnd = ev["RoundNumber"] if "RoundNumber" in ev and pd.notna(ev["RoundNumber"]) else None  # Round (ou None)
        ename = ev["EventName"]                                       # Nome do evento (fallback/log)

        try:                                                          # Tenta extrair laps
            df_laps = extract_event_race_laps(y, rnd, ename)          # Extrai voltas da sessão 'R'
            df_laps["Circuito"] = circuito                            # Marca circuito
            df_laps["EventDate"] = pd.to_datetime(ev["EventDate"]).date() if pd.notna(ev["EventDate"]) else pd.NaT
                                                                       # Data do evento como date
            df_laps["Location"] = ev["Location"]                      # Cidade/pista (Location)
            df_laps["OfficialEventName"] = ev.get("OfficialEventName", np.nan)  # Nome oficial completo
            per_circuit_laps.append(df_laps)                          # Acumula
            print(f"[OK] {circuito} {y} (Round {rnd if rnd is not None else 'n/a'}): {len(df_laps)} voltas.")
        except Exception as e:                                        # Trata erro
            print(f"[ERRO] {circuito} {y} (Round {rnd if rnd is not None else 'n/a'}): {e}")
            errors.append((circuito, y, str(e)))                      # Guarda info do erro

    # --- ETAPA 3: consolidar e salvar saídas do circuito ---
    if per_circuit_laps:                                              # Se coletou algo
        laps_circuito = pd.concat(per_circuit_laps, ignore_index=True)  # Concatena anos do circuito
        sort_cols = [c for c in ["Year","Driver","LapNumber"] if c in laps_circuito.columns]  # Colunas p/ ordenação
        if sort_cols:                                                 # Se existirem
            laps_circuito = laps_circuito.sort_values(sort_cols).reset_index(drop=True)  # Ordena

        laps_circuito.to_csv(out_csv, index=False, encoding="utf-8")  # Salva CSV
        laps_circuito.to_parquet(out_parq, index=False)               # Salva Parquet
        print(f"[SALVO] {circuito}:")                                  # Log de salvamento
        print(f"        CSV:     {out_csv}")                           # Caminho CSV
        print(f"        Parquet: {out_parq}")                          # Caminho Parquet

        all_circuits_laps.append(laps_circuito)                        # Acumula no mestre
    else:                                                              # Se nada foi extraído
        print(f"[AVISO] Nenhuma volta consolidada para '{circuito}'. Erros: {len(errors)}")  # Loga aviso

# =========================
# Dataset mestre (todos circuitos juntos)
# =========================
if all_circuits_laps:                                                 # Se houve algum sucesso
    master = pd.concat(all_circuits_laps, ignore_index=True)          # Concatena tudo

    preview_cols = [c for c in ["Circuito","Year","EventName","Driver","LapNumber","LapTime","Compound","Stint","PitInTime","PitOutTime"] if c in master.columns]
                                                                       # Colunas sugeridas para preview
    print("\n=== AMOSTRA (dataset mestre) ===")                       # Título do preview
    if preview_cols:                                                  # Se colunas existirem
        print(master[preview_cols].head(12).to_string(index=False))   # Mostra 12 linhas
    else:                                                             # Caso contrário
        print(master.head(12).to_string(index=False))                 # Mostra 12 linhas genéricas

    master_csv  = INTERIM / "ALLCIRCUITS_2022-2024_all_laps.csv"      # Caminho CSV mestre
    master_parq = INTERIM / "ALLCIRCUITS_2022-2024_all_laps.parquet"  # Caminho Parquet mestre
    master.to_csv(master_csv, index=False, encoding="utf-8")          # Salva CSV mestre
    master.to_parquet(master_parq, index=False)                       # Salva Parquet mestre

    print(f"\n[SALVO] Dataset mestre:")                               # Confirmação
    print(f"        CSV:     {master_csv}")                           # Caminho CSV
    print(f"        Parquet: {master_parq}")                          # Caminho Parquet
else:                                                                  # Se nenhum circuito produziu dados
    raise RuntimeError("Nenhuma volta foi extraída para os circuitos solicitados.")  # Erro explícito


Cache ativo em: C:\Users\pedro\iCloudDrive\Desktop\Eng. Elétrica UNESP Bauru 2020\TG\TG2\f1-tyre-strategy-simulator\notebooks\fastf1_cache
Saídas em:      C:\Users\pedro\iCloudDrive\Desktop\Eng. Elétrica UNESP Bauru 2020\TG\TG2\f1-tyre-strategy-simulator\notebooks\data\interim
[PULADO] Bahrein: carregado de Parquet existente.
[PULADO] Jeddah: carregado de Parquet existente.
[PULADO] Australia: carregado de Parquet existente.
[PULADO] Baku: carregado de Parquet existente.
[PULADO] Miami: carregado de Parquet existente.
[PULADO] Monza: carregado de Parquet existente.
[PULADO] Singapura: carregado de Parquet existente.
[PULADO] Suzuka: carregado de Parquet existente.
[PULADO] COTA: carregado de Parquet existente.
[PULADO] Mexico: carregado de Parquet existente.
[PULADO] Brasil: carregado de Parquet existente.
[PULADO] Abu Dhabi: carregado de Parquet existente.
[PULADO] Silverstone: carregado de Parquet existente.
[PULADO] Bélgica: carregado de Parquet existente.
[PULADO] Hungria: carregad

In [10]:
# === CÉLULA: Pós-processamento do DATASET MESTRE ===
from pathlib import Path
import pandas as pd

# Caminhos (usa o mesmo layout do script anterior; ajusta se necessário)
NB_DIR = Path.cwd().resolve() if Path.cwd().name.lower() == "notebooks" else Path.cwd().resolve() / "notebooks"
INTERIM = NB_DIR / "data" / "interim"
master_parq = INTERIM / "ALLCIRCUITS_2022-2024_all_laps.parquet"
master_csv  = INTERIM / "ALLCIRCUITS_2022-2024_all_laps.csv"

# --- Carrega o dataset mestre (prefere csv) ---
if master_csv.exists():
    df = pd.read_csv(master_csv)
    print(f"[LOAD] {master_csv}")
elif master_parq.exists():
    df = pd.read_parquet(master_parq)
    print(f"[LOAD] {master_parq}")
else:
    raise FileNotFoundError("Gere primeiro o dataset mestre (ALLCIRCUITS_2022-2024_all_laps.*).")

# --- Cria/normaliza LapTimeSec (segundos) ---
if "LapTimeSec" not in df.columns:
    if "LapTime" in df.columns:
        if pd.api.types.is_timedelta64_dtype(df["LapTime"]):
            df["LapTimeSec"] = df["LapTime"].dt.total_seconds()     # converte timedelta -> segundos
        else:
            df["LapTimeSec"] = pd.to_numeric(df["LapTime"], errors="coerce")  # tenta converter numérico
    else:
        df["LapTimeSec"] = pd.NA

# --- Padroniza Compound e TrackStatus para evitar perdas no filtro ---
if "Compound" in df.columns:
    df["Compound"] = df["Compound"].astype(str).str.upper()
if "TrackStatus" in df.columns:
    df["_TrackStatusStr"] = df["TrackStatus"].astype(str).str.strip()

# ========== FILTROS (ORDEM OTIMIZADA) ==========

# 0) Remover voltas marcadas como deletadas (mantém apenas Deleted == False)
if "Deleted" in df.columns:
    before = len(df)
    df = df[df["Deleted"] == False]
    after = len(df)
    print(f"Removidas {before - after} voltas com Deleted diferente de False. Restantes: {after}")
else:
    print("Coluna 'Deleted' não encontrada no DataFrame.")

# 1) Remover voltas imprecisas (IsAccurate == False)
if "IsAccurate" in df.columns:
    before = len(df)
    df = df[df["IsAccurate"]]
    after = len(df)
    print(f"Removidas {before - after} voltas com IsAccurate = False. Restantes: {after}")
else:
    print("Coluna 'IsAccurate' não encontrada no DataFrame.")

# 2) Remover linhas com tempos/setores críticos ausentes (nulos)

if "Sector1SessionTime" in df.columns:
    before = len(df)
    df = df.dropna(subset=["Sector1SessionTime"])
    after = len(df)
    print(f"Removidas {before - after} voltas com Sector1SessionTime nulo. Restantes: {after}")
else:
    print("Coluna 'Sector1SessionTime' não encontrada no DataFrame.")

if "SpeedFL" in df.columns:
    before = len(df)
    df = df.dropna(subset=["SpeedFL"])
    after = len(df)
    print(f"Removidas {before - after} voltas com SpeedFL nulo. Restantes: {after}")
else:
    print("Coluna 'SpeedFL' não encontrada no DataFrame.")

if "TyreLife" in df.columns:
    before = len(df)
    df = df.dropna(subset=["TyreLife"])
    after = len(df)
    print(f"Removidas {before - after} voltas sem TyreLife. Restantes: {after}")
else:
    print("Coluna 'TyreLife' não encontrada no DataFrame.")

# 4) Apenas compostos secos (SOFT/MEDIUM/HARD)
if "Compound" in df.columns:
    before = len(df)
    # se necessário, padronize: df["Compound"] = df["Compound"].astype(str).str.upper()
    df = df[df["Compound"].isin(["SOFT", "MEDIUM", "HARD"])]
    after = len(df)
    print(f"Removidas {before - after} voltas com pneus intermediários/molhados. Restantes: {after}")

# 5) Apenas pista verde (TrackStatus == 1)
if "TrackStatus" in df.columns:
    before = len(df)
    # auxiliar robusto para comparar com '1'
    _ts = df["TrackStatus"].astype(str).str.strip()
    mask_green = (_ts == "1") | (pd.to_numeric(df["TrackStatus"], errors="coerce") == 1)
    df = df[mask_green]
    after = len(df)
    print(f"Removidas {before - after} voltas com bandeiras/SC/VSC. Restantes: {after}")

# 6) ≤ 120% da mediana de LapTimeSec específica de cada evento (Year + EventName)
if {"LapTimeSec","Year","EventName"} <= set(df.columns):
    before_total = len(df)

    def _filter_event(g):
        med = pd.to_numeric(g["LapTimeSec"], errors="coerce").median(skipna=True)
        if pd.notna(med):
            lim = 1.2 * med
            return g[pd.to_numeric(g["LapTimeSec"], errors="coerce") <= lim]
        return g

    df = df.groupby(["Year", "EventName"], group_keys=False).apply(_filter_event)
    after_total = len(df)
    print(f"Filtradas {before_total - after_total} voltas > 120% da mediana por evento. Restantes: {after_total}")
else:
    print("Aviso: faltam colunas para o filtro de 120% por evento (LapTimeSec/Year/EventName).")


# --- Inspeção rápida ---
print("\n=== INFO PÓS-FILTRO ===")
df.info()
display(df.head())

# --- Salva dataset filtrado ---
out_parq = INTERIM / "ALLCIRCUITS_2022-2024_all_laps_FILTERED.parquet"
out_csv  = INTERIM / "ALLCIRCUITS_2022-2024_all_laps_FILTERED.csv"
df.to_parquet(out_parq, index=False)
df.to_csv(out_csv, index=False, encoding="utf-8")
print(f"\n[SALVO] Parquet: {out_parq}")
print(f"[SALVO] CSV:     {out_csv}")


C:\Users\pedro\AppData\Local\Temp\ipykernel_9588\2668396843.py:13: DtypeWarning: Columns (18,27,28) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(master_csv)


[LOAD] C:\Users\pedro\iCloudDrive\Desktop\Eng. Elétrica UNESP Bauru 2020\TG\TG2\f1-tyre-strategy-simulator\notebooks\data\interim\ALLCIRCUITS_2022-2024_all_laps.csv
Removidas 1856 voltas com Deleted diferente de False. Restantes: 47392
Removidas 6364 voltas com IsAccurate = False. Restantes: 41028
Removidas 94 voltas com Sector1SessionTime nulo. Restantes: 40934
Removidas 3 voltas com SpeedFL nulo. Restantes: 40931
Removidas 87 voltas sem TyreLife. Restantes: 40844
Removidas 2609 voltas com pneus intermediários/molhados. Restantes: 38235
Removidas 1008 voltas com bandeiras/SC/VSC. Restantes: 37227
Filtradas 0 voltas > 120% da mediana por evento. Restantes: 37227

=== INFO PÓS-FILTRO ===
<class 'pandas.core.frame.DataFrame'>
Index: 37227 entries, 1 to 49247
Data columns (total 41 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Time                37227 non-null  object 
 1   Driver              37227 non-null  object 
 2  

c:\Users\pedro\iCloudDrive\Desktop\Eng. Elétrica UNESP Bauru 2020\TG\TG2\f1-tyre-strategy-simulator\.venv\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\pedro\iCloudDrive\Desktop\Eng. Elétrica UNESP Bauru 2020\TG\TG2\f1-tyre-strategy-simulator\.venv\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\pedro\iCloudDrive\Desktop\Eng. Elétrica UNESP Bauru 2020\TG\TG2\f1-tyre-strategy-simulator\.venv\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\pedro\iCloudDrive\Desktop\Eng. Elétrica UNESP Bauru 2020\TG\TG2\f1-tyre-strategy-simulator\.venv\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out,

,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,Year,RoundNumber,EventName,SessionName,Circuito,EventDate,Location,OfficialEventName,LapTimeSec,_TrackStatusStr
1,0 days 01:06:03.288000,ALB,23,0 days 00:01:40.548000,2.0,1.0,NaN,NaN,0 days 00:00:32.027000,0 days 00:00:43.725000,...,2022,1,Bahrain Grand Prix,Race,Bahrein,2022-03-20,Sakhir,FORMULA 1 GULF AIR BAHRAIN GRAND PRIX 2022,NaN,1
2,0 days 01:07:43.952000,ALB,23,0 days 00:01:40.664000,3.0,1.0,NaN,NaN,0 days 00:00:32.056000,0 days 00:00:43.928000,...,2022,1,Bahrain Grand Prix,Race,Bahrein,2022-03-20,Sakhir,FORMULA 1 GULF AIR BAHRAIN GRAND PRIX 2022,NaN,1
3,0 days 01:09:25.078000,ALB,23,0 days 00:01:41.126000,4.0,1.0,NaN,NaN,0 days 00:00:32.050000,0 days 00:00:44.161000,...,2022,1,Bahrain Grand Prix,Race,Bahrein,2022-03-20,Sakhir,FORMULA 1 GULF AIR BAHRAIN GRAND PRIX 2022,NaN,1
4,0 days 01:11:07.381000,ALB,23,0 days 00:01:42.303000,5.0,1.0,NaN,NaN,0 days 00:00:32.792000,0 days 00:00:44.560000,...,2022,1,Bahrain Grand Prix,Race,Bahrein,2022-03-20,Sakhir,FORMULA 1 GULF AIR BAHRAIN GRAND PRIX 2022,NaN,1
5,0 days 01:12:49.089000,ALB,23,0 days 00:01:41.708000,6.0,1.0,NaN,NaN,0 days 00:00:32.220000,0 days 00:00:44.522000,...,2022,1,Bahrain Grand Prix,Race,Bahrein,2022-03-20,Sakhir,FORMULA 1 GULF AIR BAHRAIN GRAND PRIX 2022,NaN,1



[SALVO] Parquet: C:\Users\pedro\iCloudDrive\Desktop\Eng. Elétrica UNESP Bauru 2020\TG\TG2\f1-tyre-strategy-simulator\notebooks\data\interim\ALLCIRCUITS_2022-2024_all_laps_FILTERED.parquet
[SALVO] CSV:     C:\Users\pedro\iCloudDrive\Desktop\Eng. Elétrica UNESP Bauru 2020\TG\TG2\f1-tyre-strategy-simulator\notebooks\data\interim\ALLCIRCUITS_2022-2024_all_laps_FILTERED.csv


In [11]:
# === CÉLULA: Resumo estatístico e informativo do DataFrame final ===

print("\n=== RESUMO GERAL ===")
print(f"Total de linhas: {len(df)}")
print(f"Total de colunas: {len(df.columns)}")

# --- Visão geral do DataFrame ---
print("\n=== INFO ===")
df.info()

# --- Estatísticas descritivas para colunas numéricas ---
print("\n=== DESCRIBE (Numéricas) ===")
display(df.describe().T)  # .T para facilitar leitura (variáveis em linhas)

# --- Estatísticas descritivas também para colunas de texto/categóricas ---
print("\n=== DESCRIBE (Categóricas) ===")
display(df.describe(include='object').T)

# --- Frequência de valores para colunas não numéricas ---
print("\n=== FREQUÊNCIA DE VALORES (TOP 5) ===")
for col in df.columns:
    if df[col].dtype == 'object' or df[col].dtype.name == 'category' or df[col].dtype == 'bool':
        print(f"\nColuna: {col}")
        print(df[col].value_counts(dropna=False).head(5))



=== RESUMO GERAL ===
Total de linhas: 37227
Total de colunas: 41

=== INFO ===
<class 'pandas.core.frame.DataFrame'>
Index: 37227 entries, 1 to 49247
Data columns (total 41 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Time                37227 non-null  object 
 1   Driver              37227 non-null  object 
 2   DriverNumber        37227 non-null  int64  
 3   LapTime             37227 non-null  object 
 4   LapNumber           37227 non-null  float64
 5   Stint               37227 non-null  float64
 6   PitOutTime          0 non-null      object 
 7   PitInTime           0 non-null      object 
 8   Sector1Time         37227 non-null  object 
 9   Sector2Time         37227 non-null  object 
 10  Sector3Time         37227 non-null  object 
 11  Sector1SessionTime  37227 non-null  object 
 12  Sector2SessionTime  37227 non-null  object 
 13  Sector3SessionTime  37227 non-null  object 
 14  SpeedI1             30767 n

,count,mean,std,min,25%,50%,75%,max
DriverNumber,37227.0,27.179467,22.937523,1.0,11.0,21.0,44.0,81.0
LapNumber,37227.0,29.916593,16.944060,2.0,15.0,29.0,43.0,78.0
Stint,37227.0,2.043759,0.887749,1.0,1.0,2.0,3.0,6.0
SpeedI1,30767.0,260.670296,43.525115,60.0,213.0,277.0,291.0,355.0
SpeedI2,37227.0,247.043141,47.572726,115.0,196.0,249.0,295.0,343.0
SpeedFL,37227.0,262.342064,36.492471,69.0,243.0,258.0,291.0,357.0
SpeedST,34013.0,302.661218,16.296250,100.0,292.0,302.0,315.0,357.0
TyreLife,37227.0,15.191125,10.587133,2.0,7.0,13.0,20.0,78.0
TrackStatus,37227.0,1.000000,0.000000,1.0,1.0,1.0,1.0,1.0
Position,37227.0,9.596288,5.354182,1.0,5.0,9.0,14.0,20.0



=== DESCRIBE (Categóricas) ===


,count,unique,top,freq
Time,37227,37139,0 days 02:04:11.971000,2
Driver,37227,28,NOR,2007
LapTime,37227,22516,0 days 00:01:33.317000,8
PitOutTime,0,0,NaN,NaN
PitInTime,0,0,NaN,NaN
Sector1Time,37227,13337,0 days 00:00:29.021000,14
Sector2Time,37227,15770,0 days 00:00:29.905000,14
Sector3Time,37227,14405,0 days 00:00:25.734000,16
Sector1SessionTime,37227,37132,0 days 02:03:21.440000,2
Sector2SessionTime,37227,37120,0 days 01:12:17.129000,2



=== FREQUÊNCIA DE VALORES (TOP 5) ===

Coluna: Time
Time
0 days 02:04:11.971000    2
0 days 01:08:36.073000    2
0 days 02:15:35.740000    2
0 days 01:50:33.895000    2
0 days 01:56:45.062000    2
Name: count, dtype: int64

Coluna: Driver
Driver
NOR    2007
VER    1993
ALO    1976
RUS    1957
HAM    1927
Name: count, dtype: int64

Coluna: LapTime
LapTime
0 days 00:01:33.317000    8
0 days 00:01:33.447000    8
0 days 00:01:23.485000    8
0 days 00:01:32.640000    7
0 days 00:01:24.748000    7
Name: count, dtype: int64

Coluna: PitOutTime
PitOutTime
NaN    37227
Name: count, dtype: int64

Coluna: PitInTime
PitInTime
NaN    37227
Name: count, dtype: int64

Coluna: Sector1Time
Sector1Time
0 days 00:00:29.021000    14
0 days 00:00:29.917000    13
0 days 00:00:28.304000    13
0 days 00:00:30.067000    13
0 days 00:00:31.023000    13
Name: count, dtype: int64

Coluna: Sector2Time
Sector2Time
0 days 00:00:29.905000    14
0 days 00:00:30.153000    13
0 days 00:00:35.329000    12
0 days 00:00:2

In [12]:
# Garante que temos as colunas necessárias
if {"Year", "EventName", "Driver", "LapNumber"} <= set(df.columns):
    # Conta quantas voltas cada piloto completou em cada evento
    laps_por_piloto = df.groupby(["Year", "EventName", "Driver"]).size().reset_index(name="Voltas")

    # Agora agrupa por evento e calcula média e mediana do número de voltas
    resumo_evento = laps_por_piloto.groupby([ "EventName", "Year"])["Voltas"].agg(
        media_voltas="mean",
        mediana_voltas="median"
    ).reset_index()

    print("\n=== Média e Mediana de Voltas por Evento ===")
    display(resumo_evento)

else:
    print("O DataFrame precisa ter as colunas: Year, EventName, Driver e LapNumber.")


=== Média e Mediana de Voltas por Evento ===


,EventName,Year,media_voltas,mediana_voltas
0,Abu Dhabi Grand Prix,2022,49.450000,50.5
1,Abu Dhabi Grand Prix,2023,51.450000,52.0
2,Abu Dhabi Grand Prix,2024,44.947368,47.0
3,Australian Grand Prix,2022,41.684211,44.0
4,Australian Grand Prix,2023,39.631579,44.0
5,Australian Grand Prix,2024,44.578947,49.0
6,Azerbaijan Grand Prix,2022,36.600000,41.0
7,Azerbaijan Grand Prix,2023,40.100000,42.0
8,Azerbaijan Grand Prix,2024,42.500000,44.5
9,Bahrain Grand Prix,2022,42.600000,42.0


In [13]:
# === CÉLULA: Gráficos por circuito ===
import matplotlib.pyplot as plt
import seaborn as sns

# Garante que temos Circuito, TyreLife, LapTimeSec, Compound, TrackTemp e Year
if "Circuito" in df.columns:
    circuitos = sorted(df["Circuito"].dropna().unique())
    print(f"Gerando gráficos para {len(circuitos)} circuitos...")

    for circuito in circuitos:
        subset = df[df["Circuito"] == circuito]
        print(f"\n=== {circuito} ===")

        # # ----- Gráfico 1: Degradação - tempo da volta vs idade do pneu -----
        # if {"TyreLife","LapTimeSec","Compound"} <= set(subset.columns):
        #     plt.figure(figsize=(8,5))
        #     sns.scatterplot(data=subset, x="TyreLife", y="LapTimeSec", hue="Compound")
        #     mediana_lap = subset["LapTimeSec"].median(skipna=True)
        #     if pd.notna(mediana_lap):
        #         plt.axhline(mediana_lap, color='red', linestyle='--', label=f'Mediana ({mediana_lap:.2f}s)')
        #     plt.title(f"{circuito} — Degradação: tempo da volta vs idade do pneu")
        #     plt.xlabel("Voltas com o mesmo pneu (TyreLife)")
        #     plt.ylabel("Tempo da volta (s)")
        #     plt.legend(title="Composto")
        #     plt.tight_layout()
        #     plt.show()

        # ----- Gráfico 2: Relação tempo vs temperatura da pista -----
        if {"TrackTemp","LapTimeSec","Year"} <= set(subset.columns) and subset["TrackTemp"].notna().any():
            plt.figure(figsize=(8,5))
            sns.scatterplot(data=subset, x="TrackTemp", y="LapTimeSec", hue="Year", palette="viridis")
            mediana_lap = subset["LapTimeSec"].median(skipna=True)
            if pd.notna(mediana_lap):
                plt.axhline(mediana_lap, color='red', linestyle='--', label=f'Mediana ({mediana_lap:.2f}s)')
            plt.title(f"{circuito} — Relação: tempo de volta vs temperatura da pista")
            plt.xlabel("Temperatura da pista (°C)")
            plt.ylabel("Tempo da volta (s)")
            plt.legend(title="Ano")
            plt.tight_layout()
            plt.show()
        else:
            print("⚠️ Sem dados suficientes para gráfico TrackTemp vs LapTimeSec")
else:
    print("A coluna 'Circuito' não está no DataFrame; certifique-se de ter essa informação antes de rodar esta célula.")


Gerando gráficos para 16 circuitos...

=== Abu Dhabi ===
⚠️ Sem dados suficientes para gráfico TrackTemp vs LapTimeSec

=== Australia ===
⚠️ Sem dados suficientes para gráfico TrackTemp vs LapTimeSec

=== Bahrein ===
⚠️ Sem dados suficientes para gráfico TrackTemp vs LapTimeSec

=== Baku ===
⚠️ Sem dados suficientes para gráfico TrackTemp vs LapTimeSec

=== Brasil ===
⚠️ Sem dados suficientes para gráfico TrackTemp vs LapTimeSec

=== Bélgica ===
⚠️ Sem dados suficientes para gráfico TrackTemp vs LapTimeSec

=== COTA ===
⚠️ Sem dados suficientes para gráfico TrackTemp vs LapTimeSec

=== Hungria ===
⚠️ Sem dados suficientes para gráfico TrackTemp vs LapTimeSec

=== Jeddah ===
⚠️ Sem dados suficientes para gráfico TrackTemp vs LapTimeSec

=== Mexico ===
⚠️ Sem dados suficientes para gráfico TrackTemp vs LapTimeSec

=== Miami ===
⚠️ Sem dados suficientes para gráfico TrackTemp vs LapTimeSec

=== Monza ===
⚠️ Sem dados suficientes para gráfico TrackTemp vs LapTimeSec

=== Mônaco ===
⚠️ Sem 